<a href="https://colab.research.google.com/github/anushka123416-cell/food-delivery-order-analysis/blob/main/food_analysis_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
from google.colab import files

uploaded = files.upload()

In [6]:
import numpy as np
import pandas as pd

# Dataset load karein
df = pd.read_csv("order_history_kaggle_data.csv", on_bad_lines="skip")
df.columns = df.columns.str.strip()

print("Total rows & columns:", df.shape)
print("Columns:", list(df.columns))
print("\nMissing values:\n", df.isnull().sum())

# User se puchenge kitni rows dekhni hain
n = int(input("\nKitne top orders dekhne hain? "))
print(f"\nTop {n} orders:")
print(df.head(n))

# Numeric columns par NumPy se basic stats
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()

if num_cols:
  print("\nNumeric Summary (Mean, Median, Std):")
  for col in num_cols:
    arr = df[col].dropna().to_numpy()
    print(
        f"  {col}: Mean={np.mean(arr):.2f}, Median={np.median(arr):.2f},"
        f" Std={np.std(arr):.2f}"
    )


Total rows & columns: (21321, 29)
Columns: ['Restaurant ID', 'Restaurant name', 'Subzone', 'City', 'Order ID', 'Order Placed At', 'Order Status', 'Delivery', 'Distance', 'Items in order', 'Instructions', 'Discount construct', 'Bill subtotal', 'Packaging charges', 'Restaurant discount (Promo)', 'Restaurant discount (Flat offs, Freebies & others)', 'Gold discount', 'Brand pack discount', 'Total', 'Rating', 'Review', 'Cancellation / Rejection reason', 'Restaurant compensation (Cancellation)', 'Restaurant penalty (Rejection)', 'KPT duration (minutes)', 'Rider wait time (minutes)', 'Order Ready Marked', 'Customer complaint tag', 'Customer ID']

Missing values:
 Restaurant ID                                             0
Restaurant name                                           0
Subzone                                                   0
City                                                      0
Order ID                                                  0
Order Placed At                    

In [7]:
# 1. Order Status summary
print("=== Order Status Count ===")
print(df["Order Status"].value_counts())

# 2. NumPy np.where se Order Size category (Small / Medium / Large)
subtotal = pd.to_numeric(df["Bill subtotal"], errors="coerce").fillna(0)
df["Order_Category"] = np.where(
    subtotal < 200,
    "Small (<200)",
    np.where(subtotal <= 500, "Medium (200-500)", "Large (>500)"),
)
print("\n=== Order Category Count ===")
print(df["Order_Category"].value_counts())

# 3. User se Restaurant name lekar search (Spotify artist filter ki tarah)
rest_query = input(
    "\nKonsa restaurant search karna hai? (e.g. Swaad): "
).strip()
if rest_query:
  res_df = df[
      df["Restaurant name"]
      .astype(str)
      .str.contains(rest_query, case=False, na=False)
  ]
  print(f"\n'{rest_query}' ke liye {len(res_df)} orders mile:")
  print(
      res_df[[
          "Order ID",
          "Restaurant name",
          "Items in order",
          "Bill subtotal",
          "Total",
          "Order Status",
      ]].head(5)
  )

# 4. User se Food Item lekar search
item_query = input("\nKonsa item search karna hai? (e.g. Chicken): ").strip()
if item_query:
  item_df = df[
      df["Items in order"]
      .astype(str)
      .str.contains(item_query, case=False, na=False)
  ]
  print(f"\n'{item_query}' wale {len(item_df)} orders mile:")
  print(
      item_df[["Order ID", "Restaurant name", "Items in order", "Total"]].head(
          5
      )
  )

=== Order Status Count ===
Order Status
Delivered           21131
Rejected              158
Returned               25
Return cancelled        3
Picked up               3
Timed out               1
Name: count, dtype: int64

=== Order Category Count ===
Order_Category
Large (>500)        14642
Medium (200-500)     6519
Small (<200)          160
Name: count, dtype: int64

Konsa restaurant search karna hai? (e.g. Swaad): swaad

'swaad' ke liye 6332 orders mile:
     Order ID Restaurant name  \
0  6168884918           Swaad   
1  6170707559           Swaad   
2  6169375019           Swaad   
3  6151677434           Swaad   
4  6167540897           Swaad   

                                      Items in order  Bill subtotal    Total  \
0  1 x Grilled Chicken Jamaican Tender, 1 x Grill...          715.0   666.75   
1  1 x Peri Peri Fries, 1 x Fried Chicken Angara ...         1179.0  1054.20   
2              1 x Bone in Peri Peri Grilled Chicken          310.0   241.50   
3  1 x Fried Chicke

In [8]:
# Total Discount aur Net Savings ka simple NumPy analysis
promo = pd.to_numeric(
    df["Restaurant discount (Promo)"], errors="coerce"
).fillna(0)
flat_off = pd.to_numeric(
    df["Restaurant discount (Flat offs, Freebies & others)"], errors="coerce"
).fillna(0)
gold = pd.to_numeric(df["Gold discount"], errors="coerce").fillna(0)

total_discount = (promo + flat_off + gold).to_numpy()
print("=== Total Discount Analysis (NumPy) ===")
print("Average Discount per Order:", round(float(np.mean(total_discount)), 2))
print("Max Discount Given:", round(float(np.max(total_discount)), 2))
print("Total Discount Across All Orders:", round(float(np.sum(total_discount)), 2))

=== Total Discount Analysis (NumPy) ===
Average Discount per Order: 96.99
Max Discount Given: 7787.0
Total Discount Across All Orders: 2067838.55
